# 01 — Data Audit

This notebook performs the initial quality and structure assessment of the Bank Account Fraud (BAF) base dataset. The goal is to understand the dataset before any modeling, identify data-quality issues, and define a leakage-safe temporal evaluation strategy.

## Load Data

Load the BAF `Base.csv` dataset and inspect its overall dimensions and sample records.

In [4]:
#!/usr/bin/env python3

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"D:\bank-account-fraud-decision-system"
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

data_path = RAW_DATA_DIR / "Base.csv"

df = pd.read_csv(data_path)

print("Shape:", df.shape)
display(df.head())

Shape: (1000000, 32)


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


## Dataset Structure

Inspect data types, feature names, target distribution, duplicate rows, and monthly application counts.

In [5]:
df.info()
print(df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 32 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   fraud_bool                        1000000 non-null  int64  
 1   income                            1000000 non-null  float64
 2   name_email_similarity             1000000 non-null  float64
 3   prev_address_months_count         1000000 non-null  int64  
 4   current_address_months_count      1000000 non-null  int64  
 5   customer_age                      1000000 non-null  int64  
 6   days_since_request                1000000 non-null  float64
 7   intended_balcon_amount            1000000 non-null  float64
 8   payment_type                      1000000 non-null  str    
 9   zip_count_4w                      1000000 non-null  int64  
 10  velocity_6h                       1000000 non-null  float64
 11  velocity_24h                      1000000 non-nul

In [6]:
# Target Variable Analysis

print("Target counts:")
print(df["fraud_bool"].value_counts())

print("\nTarget percentages:")
print(
    df["fraud_bool"]
    .value_counts(normalize=True)
    .mul(100)
    .round(4)
)

Target counts:
fraud_bool
0    988971
1     11029
Name: count, dtype: int64

Target percentages:
fraud_bool
0    98.8971
1     1.1029
Name: proportion, dtype: float64


In [7]:
print(
    "Exact duplicate rows:",
    df.duplicated().sum()
)

Exact duplicate rows: 0


In [8]:
print(
    df["month"]
    .value_counts()
    .sort_index()
)

month
0    132440
1    127620
2    136979
3    150936
4    127691
5    119323
6    108168
7     96843
Name: count, dtype: int64


## Hidden Missing-Value Audit

The dataset contains some sentinel values such as `-1` that represent missing information for selected features. Negative values are audited carefully rather than being treated as missing automatically.

In [9]:
# Check negative/sentinel values in numeric columns

numeric_cols = df.select_dtypes(
    include=["number"]
).columns.tolist()

sentinel_summary = []

for col in numeric_cols:
    negative_count = (df[col] < 0).sum()

    if negative_count > 0:
        sentinel_summary.append({
            "feature": col,
            "negative_count": negative_count,
            "negative_pct": round(
                negative_count / len(df) * 100,
                2
            ),
            "min_value": df[col].min()
        })

sentinel_summary = pd.DataFrame(
    sentinel_summary
).sort_values(
    "negative_pct",
    ascending=False
)

display(sentinel_summary)

,feature,negative_count,negative_pct,min_value
2,intended_balcon_amount,742523,74.25,-15.530555
0,prev_address_months_count,712920,71.29,-1.000000
5,bank_months_count,253635,25.36,-1.000000
4,credit_risk_score,14445,1.44,-170.000000
1,current_address_months_count,4254,0.43,-1.000000
6,session_length_in_minutes,2015,0.20,-1.000000
7,device_distinct_emails_8w,359,0.04,-1.000000
3,velocity_6h,44,0.00,-170.603072


In [10]:
unique_counts = (
    df.nunique()
    .sort_values()
)

display(unique_counts.head(10))

device_fraud_count           1
fraud_bool                   2
keep_alive_session           2
foreign_request              2
phone_home_valid             2
has_other_cards              2
email_is_free                2
source                       2
phone_mobile_valid           2
device_distinct_emails_8w    4
dtype: int64

## Temporal Fraud Drift

Fraud prevalence is examined across monthly periods to determine whether the target distribution changes over time.

In [11]:
#Monthly Fraud Analysis

monthly_fraud = (
    df.groupby("month")["fraud_bool"]
    .agg(
        applications="count",
        fraud_cases="sum",
        fraud_rate="mean"
    )
)

monthly_fraud["fraud_rate_pct"] = (
    monthly_fraud["fraud_rate"] * 100
).round(4)

display(monthly_fraud)

,applications,fraud_cases,fraud_rate,fraud_rate_pct
month,,,,
0,132440,1500,0.011326,1.1326
1,127620,1198,0.009387,0.9387
2,136979,1198,0.008746,0.8746
3,150936,1392,0.009222,0.9222
4,127691,1452,0.011371,1.1371
5,119323,1411,0.011825,1.1825
6,108168,1450,0.013405,1.3405
7,96843,1428,0.014746,1.4746


In [12]:
# Check for -1 values in numeric columns

minus_one_summary = []

for col in numeric_cols:
    count = (df[col] == -1).sum()

    if count > 0:
        minus_one_summary.append({
            "feature": col,
            "minus_one_count": count,
            "minus_one_pct": round(
                count / len(df) * 100,
                3
            )
        })

minus_one_summary = pd.DataFrame(
    minus_one_summary
).sort_values(
    "minus_one_pct",
    ascending=False
)

display(minus_one_summary)

,feature,minus_one_count,minus_one_pct
0,prev_address_months_count,712920,71.292
3,bank_months_count,253635,25.364
1,current_address_months_count,4254,0.425
4,session_length_in_minutes,2015,0.201
2,credit_risk_score,488,0.049
5,device_distinct_emails_8w,359,0.036


In [13]:
#Cardinality Analysis

display(
    df.nunique()
    .sort_values()
    .head(20)
)

device_fraud_count                   1
fraud_bool                           2
keep_alive_session                   2
foreign_request                      2
phone_home_valid                     2
has_other_cards                      2
email_is_free                        2
source                               2
phone_mobile_valid                   2
device_distinct_emails_8w            4
device_os                            5
payment_type                         5
employment_status                    7
housing_status                       7
month                                8
income                               9
customer_age                         9
proposed_credit_limit               12
bank_months_count                   33
date_of_birth_distinct_emails_4w    40
dtype: int64

In [14]:
#Categorical columns analysis

categorical_cols = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Categorical columns:", categorical_cols)

for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))       #dropna=False includes NaN values

Categorical columns: ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']

payment_type
payment_type
AB    370554
AA    258249
AC    252071
AD    118837
AE       289
Name: count, dtype: int64

employment_status
employment_status
CA    730252
CB    138288
CF     44034
CC     37758
CD     26522
CE     22693
CG       453
Name: count, dtype: int64

housing_status
housing_status
BC    372143
BB    260965
BA    169675
BE    169135
BD     26161
BF      1669
BG       252
Name: count, dtype: int64

source
source
INTERNET    992952
TELEAPP       7048
Name: count, dtype: int64

device_os
device_os
other        342728
linux        332712
windows      263506
macintosh     53826
x11            7228
Name: count, dtype: int64


## Data-Quality Cleaning Policy

The constant `device_fraud_count` feature is removed because it contains no variation. Confirmed `-1` sentinel values are converted to missing values, while missingness indicator features are retained so the model can learn whether missingness itself is informative.

No rows are removed during this stage.

In [23]:
# Drop constant feature
df_clean = df.drop(
    columns=["device_fraud_count"]
).copy()


# Create missing value indicator columns for sentinel values
sentinel_features = [
    "prev_address_months_count",
    "current_address_months_count",
    "bank_months_count",
    "session_length_in_minutes",
    "device_distinct_emails_8w"
]

for col in sentinel_features:
    df_clean[f"{col}_missing"] = (
        df_clean[col] == -1
    ).astype(int)

In [24]:
# Replace sentinel values with NaN
for col in sentinel_features:
    df_clean.loc[
        df_clean[col] == -1,
        col
    ] = np.nan

In [27]:
# Check for missing values after cleaning
missing_after_cleaning = (
    df_clean
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_after_cleaning[
        missing_after_cleaning > 0
    ]
)

prev_address_months_count       712920
bank_months_count               253635
current_address_months_count      4254
session_length_in_minutes         2015
device_distinct_emails_8w          359
dtype: int64

In [26]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

Original shape: (1000000, 32)
Cleaned shape: (1000000, 36)


## Temporal Evaluation Strategy

A chronological split is used instead of a random split:

- Training: months 0–5
- Validation: month 6
- Test: month 7

This simulates a realistic deployment setting in which a model is trained on historical applications and evaluated on future applications.

In [20]:
#Temporal train-validation-test split
train_df = df_clean[
    df_clean["month"].between(0, 5)
].copy()

val_df = df_clean[
    df_clean["month"] == 6
].copy()

test_df = df_clean[
    df_clean["month"] == 7
].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (794989, 36)
Validation shape: (108168, 36)
Test shape: (96843, 36)


In [21]:
#Split wise fraud rate percentage
def show_fraud_rate(name, data):
    fraud_rate = data["fraud_bool"].mean() * 100

    print(
        f"{name}: "
        f"{len(data):,} rows | "
        f"Fraud rate = {fraud_rate:.4f}%"
    )

show_fraud_rate("Train", train_df)
show_fraud_rate("Validation", val_df)
show_fraud_rate("Test", test_df)

Train: 794,989 rows | Fraud rate = 1.0253%
Validation: 108,168 rows | Fraud rate = 1.3405%
Test: 96,843 rows | Fraud rate = 1.4746%


In [22]:
#Display unique months for each split
print(
    "Train months:",
    sorted(train_df["month"].unique())
)

print(
    "Validation months:",
    sorted(val_df["month"].unique())
)

print(
    "Test months:",
    sorted(test_df["month"].unique())
)

Train months: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Validation months: [np.int64(6)]
Test months: [np.int64(7)]


## Audit Summary

The dataset contains 1,000,000 applications with a highly imbalanced fraud target. No exact duplicate rows were found. Several features use sentinel-coded missing values, which were exposed as explicit missing values while retaining missingness indicators.

Fraud prevalence increases from approximately 1.03% in the training period to 1.34% in validation and 1.47% in the final test period, confirming the importance of chronological evaluation.

The cleaned dataset is therefore suitable for exploratory signal analysis and subsequent fraud-risk modeling.